# ==============================================================================
# 🕵️ ENTERPRISE MISINFORMATION FILTER: FAKE NEWS DETECTION VIA MULTINOMIAL NAIVE BAYES
# ==============================================================================
### Core Principles:
Detecting disingenuous news articles using term occurrence patterns and Bayes' theorem.
Multinomial Naive Bayes computes:
$$\log P(C_{\text{fake}} \mid X) = \log P(C_{\text{fake}}) + \sum_{i=1}^d x_i \log p_{\text{fake}, i}$$

Sensationalist and fraudulent journalism relies on distinct vocabulary frequencies.
Laplace smoothing ensures unseen buzzwords do not collapse log-likelihoods to $-\infty$.


In [1]:
# STEP 1: IMPORTS & SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Misinformation Classification Environment ready.")


✅ STEP 1: Misinformation Classification Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Fake and True News datasets (44,898 total articles).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
fake_df = pd.read_csv('data/fake_news/fake.csv')
true_df = pd.read_csv('data/fake_news/true.csv')

fake_df['label'] = 1  # 1 = Fake
true_df['label'] = 0  # 0 = Real/True

# Sample balanced subset for efficient processing
fake_sample = fake_df.sample(n=6000, random_state=42)
true_sample = true_df.sample(n=6000, random_state=42)
df = pd.concat([fake_sample, true_sample]).sample(frac=1.0, random_state=42).reset_index(drop=True)

df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(f"📊 Balanced Dataset Shape: {df.shape[0]:,} articles")
print(f"Class Balance: {df['label'].value_counts().to_dict()}")
df[['title', 'label']].head(3)


📊 Balanced Dataset Shape: 12,000 articles
Class Balance: {1: 6000, 0: 6000}


,title,label
0,White Trump Supporter Violently Attacks Musli...,1
1,Mexico to cut share of refined sugar sent to U...,0
2,"Hooded Men Spray-Paint Racial Slurs, Pro-Trum...",1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Constructing document content and auditing data hygiene.


In [3]:
# STEP 4 & 5: TEXT COMBINATION & HYGIENE
df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
X = df['full_text']
y = df['label']

print(f"Null texts: {X.isnull().sum()}")
print(f"Label Distribution:\n{y.value_counts(normalize=True).round(4) * 100}")


Null texts: 0
Label Distribution:
label
1    50.0
0    50.0
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
Creating balanced 80/20 train/test partitions.


In [4]:
# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Documents: {len(X_train):,}")
print(f"Testing Documents : {len(X_test):,}")


Training Documents: 9,600
Testing Documents : 2,400


## ⚙️ STEP 7: NLP VECTORIZATION PIPELINE
`TfidfVectorizer` with max 12,000 features, English stop words, and sublinear TF scaling.


In [5]:
# STEP 7: TFIDF SETUP
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=12000,
    sublinear_tf=True
)
print("✅ STEP 7: TF-IDF Misinformation Vectorizer configured.")


✅ STEP 7: TF-IDF Misinformation Vectorizer configured.


## ⚠️ STEP 8: BASELINE PIPELINE EXECUTION
Fitting MultinomialNB with Laplace smoothing $\alpha=0.5$.


In [6]:
# STEP 8: BASELINE PIPELINE
pipe_base = Pipeline([
    ('tfidf', tfidf),
    ('mnb', MultinomialNB(alpha=0.5))
])
pipe_base.fit(X_train, y_train)

y_pred_base = pipe_base.predict(X_test)
print(f"Baseline MultinomialNB Test Accuracy : {accuracy_score(y_test, y_pred_base)*100:.2f}%")
print(f"Baseline MultinomialNB ROC-AUC       : {roc_auc_score(y_test, pipe_base.predict_proba(X_test)[:, 1])*100:.2f}%")


Baseline MultinomialNB Test Accuracy : 95.71%


Baseline MultinomialNB ROC-AUC       : 99.04%


## 🎯 STEP 9: TUNING SMOOTHING PARAMETER $\alpha$
Optimizing $\alpha$ via 5-Fold Stratified Cross-Validation for misinformation detection.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'mnb__alpha': np.logspace(-3, 1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    pipe_base,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Alpha Parameter: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV F1-Score: {grid.best_score_*100:.2f}%")


🏆 Best Alpha Parameter: {'mnb__alpha': np.float64(0.001)}
🎯 Best 5-Fold CV F1-Score: 96.15%


## ⚔️ STEP 10: ALGORITHM DUEL (MultinomialNB vs Logistic Regression)
Comparing generative MultinomialNB with discriminative Logistic Regression.


In [8]:
# STEP 10: MODEL BENCHMARK
pipe_lr = Pipeline([
    ('tfidf', tfidf),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)

y_pred_nb = best_model.predict(X_test)
y_pred_lr = pipe_lr.predict(X_test)

print("="*65)
print(f"MultinomialNB (Tuned) -> Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}% | ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression   -> Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}% | ROC-AUC: {roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])*100:.2f}%")
print("="*65)


MultinomialNB (Tuned) -> Accuracy: 96.33% | ROC-AUC: 99.26%


Logistic Regression   -> Accuracy: 98.62% | ROC-AUC: 99.92%


## 📊 STEP 11: EVALUATION REPORT & CONFUSION MATRIX
Detailed performance diagnostics on unseen news articles.


In [9]:
# STEP 11: EVALUATION
print("📋 FAKE NEWS FILTER CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_nb, target_names=['Real/Verified News', 'Fake/Fabricated News']))

cm = confusion_matrix(y_test, y_pred_nb)
print(f"Confusion Matrix:\n{cm}")


📋 FAKE NEWS FILTER CLASSIFICATION REPORT:
                      precision    recall  f1-score   support

  Real/Verified News       0.96      0.96      0.96      1200
Fake/Fabricated News       0.96      0.96      0.96      1200

            accuracy                           0.96      2400
           macro avg       0.96      0.96      0.96      2400
        weighted avg       0.96      0.96      0.96      2400

Confusion Matrix:
[[1158   42]
 [  46 1154]]


## 💾 STEP 12 & 13: PRODUCTION EXPORT & LIVE INFERENCE TEST
Serializing production pipeline and running real-time fake news audit.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/fake_news_multinomial_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_headline = [
    "SHOCKING DISCOVERY: Secret alien base uncovered under the Pentagon reveals shocking truth politicians do not want you to know!"
]

t0 = time.perf_counter()
pred_fake = loaded_pipe.predict(sample_headline)[0]
fake_prob = loaded_pipe.predict_proba(sample_headline)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🕵️ LIVE MISINFORMATION SMOKE TEST:")
print(f"   Assessment       : {'🚨 FABRICATED / FAKE NEWS' if pred_fake == 1 else '✅ VERIFIED NEWS'}")
print(f"   Fabrication Risk : {fake_prob*100:.2f}%")
print(f"   Inference Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/fake_news_multinomial_nb_pipeline.joblib (849,505 bytes)

🕵️ LIVE MISINFORMATION SMOKE TEST:
   Assessment       : 🚨 FABRICATED / FAKE NEWS
   Fabrication Risk : 99.60%
   Inference Latency: 2.252 ms (SLA < 2.0ms: CHECK)
